In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

from FEX.utils import fex
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..')) 
if project_root not in sys.path:
    sys.path.append(project_root)

ImportError: cannot import name 'fex' from 'FEX.models' (/home/hudson/GraphFEX/FEX/models/__init__.py)

In [ ]:
from generate_data import make_arni_adjacency, make_data, make_normalized_adjacency
timesteps=10000
adj_matrix = make_arni_adjacency(100, 5)
data, derivatives = make_data(num_samples=timesteps, adjacency=adj_matrix, coupling=1.0)

In [ ]:
dimx_fex = fex.CoupledFEX('depth_1_tree_config', 'depth_2_tree_config', 0, controller_epochs = 200, inter_lr=0.02, self_lr=0.02, num_fex_epochs = 80, finetune_epochs=5000, finetune_lr=1e-4, poolsize=8, device=device, expression_threshold=0.01)
fex_ops = [0]
inter_ops = [0,6,0]
leaf_dim = data.shape[2]
dimx_fex.init(fex_ops, inter_ops, leaf_dim)
dimx_fex.finetune(data, derivatives, adj_matrix)

In [ ]:
print(dimx_fex)

In [ ]:
dimx_fex = fex.CoupledFEX('depth_1_tree_config', 'depth_2_tree_config', 0, controller_epochs = 200, inter_lr=0.02, self_lr=0.02, num_fex_epochs = 80, finetune_epochs=10000, finetune_lr=1e-4, poolsize=8, device=device, expression_threshold=0.01)
fex_ops = [0]
inter_ops = [0,6,6]
leaf_dim = data.shape[2]
dimx_fex.init(fex_ops, inter_ops, leaf_dim)
dimx_fex.finetune(data, derivatives, adj_matrix)

In [ ]:
print(dimx_fex)

In [ ]:
dimx_fex = fex.CoupledFEX('depth_2_tree_config', 'depth_2_tree_config', 0, controller_epochs = 100, inter_lr=0.02, self_lr=0.02, num_fex_epochs = 80, finetune_epochs=5000, finetune_lr=1e-4, poolsize=8, device=device, expression_threshold=0.01)
dimx_fex.fit(data, derivatives, adj_matrix, num_workers=5)

In [ ]:
print(dimx_fex)

In [ ]:
pred_traj = torch.zeros_like(data, device=device)
pred_traj[0] = data[0]
for t in range(timesteps - 1):
    pred_traj[t + 1] = pred_traj[t] + dimx_fex.predict(pred_traj[t], adj_matrix)

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(12, 6))
plt.plot(data.cpu().numpy()[:, :, 0], label='True Trajectory', color='blue')
#plt.plot(pred_traj.cpu().numpy()[:200, 0], label='Predicted Trajectory', color='orange', linestyle='--')
plt.xlabel('Time Step')
plt.ylabel('State Value')
plt.title('True vs Predicted Trajectory for Node 0')
#plt.legend()

In [ ]:
import matplotlib.animation as animation

fig, ax = plt.subplots(figsize=(4, 4))
z = np.exp(1j * data[0, :, 0].numpy())
x = np.real(z)
y = np.imag(z)
#x_pred = np.real(np.exp(1j * pred_traj[0, :, 0].detach().cpu().numpy()))
#y_pred = np.imag(np.exp(1j * pred_traj[0, :, 0].detach().cpu().numpy()))
vel = derivatives[0, :, 0].numpy()
scat_true = ax.scatter(x, y, c=vel, cmap='viridis', vmin=vel.min(), vmax=vel.max())
#scat_pred = ax.scatter(x_pred, y_pred, alpha=0.5, color='red')
ax.set_xticks([])
ax.set_yticks([])
ax.plot(np.cos(np.linspace(0, 2*np.pi, 100)), np.sin(np.linspace(0, 2*np.pi, 100)), color='black', linestyle='--')

def update(frame):
    z = np.exp(1j * data[frame, :, 0].numpy())
    x = np.real(z)
    y = np.imag(z)
    #z_pred = np.exp(1j * pred_traj[frame, :, 0].detach().cpu().numpy())
    #x_pred = np.real(z_pred)
    #y_pred = np.imag(z_pred)
    true_data = np.stack((x, y)).T
    #pred_data = np.stack((x_pred, y_pred)).T
    scat_true.set_offsets(true_data)
    #scat_pred.set_offsets(pred_data)

    vel = derivatives[frame, :, 0].numpy()
    scat_true.set_array(vel)
    return (scat_true,)

plt.xlabel(r'$x = \cos(\theta)$')
plt.ylabel(r'$y = \sin(\theta)$')
plt.title(r'Kuramoto Oscillator Dynamics')
plt.legend(['True Dynamics'], loc='upper right')
frames = np.arange(0, data.shape[0], 20)
ani = animation.FuncAnimation(fig=fig, func=update, frames=frames, interval=100, blit=True)
plt.close(fig)
from IPython.display import HTML
HTML(ani.to_jshtml())